In [ ]:
import scanpy as sc

adata=sc.read_h5ad('beta_T2D.h5ad')
adata

In [ ]:
# 假设你的数据是 adata
# 1. 标记线粒体基因 (通常以 'MT-' 开头)
adata.var['mt'] = adata.var_names.str.startswith('MT-') 

# 2. 计算 QC 指标
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

# 3. 过滤细胞和基因
# 过滤掉检测到基因数少于 200 的细胞，以及在少于 3 个细胞中表达的基因
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

# 4. 根据线粒体占比过滤 (通常阈值设为 5% 或 10%)
adata = adata[adata.obs.pct_counts_mt < 5, :]

print(f"QC 后的数据规模: {adata.n_obs} 细胞 x {adata.n_vars} 基因")

In [ ]:
# 备份原始计数数据（可选，用于某些后期分析）
adata.layers["counts"] = adata.X.copy()

# 1. 标准化：使每个细胞的总计数达到 10,000
sc.pp.normalize_total(adata, target_sum=1e4)

# 2. 对数转换：将数据转换为 log(x+1)
sc.pp.log1p(adata)

In [ ]:
# 筛选高变基因
# batch_key 参数可选，如果你的数据包含多个样本，建议填入 batch 字段以减少批次效应影响
sc.pp.highly_variable_genes(adata,n_top_genes=2000)

# 绘制高变基因图表
sc.pl.highly_variable_genes(adata)

# 只保留高变基因（或者将其标记出来，在后续 PCA 中使用）
# adata = adata[:, adata.var.highly_variable] # 如果只想保留 HVG，取消此行注释

In [ ]:
adata_hvg=adata[:,adata.var['highly_variable']]
adata_hvg

In [ ]:
adata.obs['development_stage'].isin(['34-year-old stage','35-year-old stage','37-year-old stage','38-year-old stage'])

adata.obs['disease'].isin(['type 2 diabetes mellitus','normal']).sum()

index=(adata.obs['sex']=='female') & (adata.obs['disease'].isin(['type 2 diabetes mellitus','normal'])) & (adata.obs['development_stage'].isin(['34-year-old stage','35-year-old stage','37-year-old stage','38-year-old stage']))

index.sum()

In [ ]:
processed_adata=adata_hvg[index]
processed_adata.obs['cell_type']

processed_adata.write_h5ad('beta_T2D_processed.h5ad')

In [ ]:
processed_adata.obs['cell_type']

In [ ]:
processed_adata.var['feature_name']

In [ ]:
import scanpy as sc

adata=sc.read_h5ad('beta_T2D_processed.h5ad')
adata

In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['pdf.fonttype'] = 42
# Count the occurrences of each diabetes status
status_counts = adata.obs['disease'].value_counts()

# Create a bar plot
plt.figure(figsize=(6, 4))
sns.barplot(x=status_counts.index, y=status_counts.values, palette='Set3')
plt.xlabel('Cell Conditions', size=18)
plt.ylabel('Counts of Cells', size=18)
plt.yticks([0,max(status_counts.values)//2,max(status_counts.values)],size=14)
plt.xticks(size=14)
sns.despine()
plt.savefig("beta_T2D_disease_status_counts.pdf", bbox_inches="tight")
plt.show()